In [2]:
from dataclasses import dataclass
from typing import Optional
import pandas as pd

In [4]:
@dataclass
class ETLConfig:
    logs_path: str
    notes_path: str
    course_column: str = "contexte"
    course_value: Optional[str] = None
    datetime_column: str = "heure"
    id_column: str = "pseudo"

In [5]:

class ETLPipeline:
    def __init__(self, config: ETLConfig):
        self.config = config
        self.logs_df: Optional[pd.DataFrame] = None
        self.notes_df: Optional[pd.DataFrame] = None
        self.result_df: Optional[pd.DataFrame] = None

In [6]:
 # ---------- Extraction des données  ----------
    def extract(self) -> None:
        self.logs_df = pd.read_csv(self.config.logs_path)
        self.notes_df = pd.read_csv(self.config.notes_path)


IndentationError: unexpected indent (1010456690.py, line 2)

In [7]:
   
    # ----------nettoyage des données  ----------
    def clean(self) -> None:
        for df in [self.logs_df, self.notes_df]:
            if df is None:
                continue
            # enlever lignes entièrement vides
            df.dropna(how="all", inplace=True)
            # enlever doublons exacts
            df.drop_duplicates(inplace=True)
            # éventuellement : supprimer espaces autour des chaînes
            for col in df.select_dtypes(include="object").columns:
                df[col] = df[col].str.strip()

In [8]:

    # ---------- Transformation des données ----------
    def transform(self) -> None:
        # parser les dates
        self.logs_df[self.config.datetime_column] = pd.to_datetime(
            self.logs_df[self.config.datetime_column],
            errors="coerce"
        )


In [9]:

        # filtrage 
        if self.config.course_value is not None:
            self.logs_df = self.logs_df[
                self.logs_df[self.config.course_column] == self.config.course_value
                ]

NameError: name 'self' is not defined

In [10]:


        # fusionner logs + notes sur pseudo
        self.result_df = pd.merge(
            self.logs_df,
            self.notes_df,
            on=self.config.id_column,
            how="inner"
        )

        # on peut éventuellement supprimer les lignes dont la date est manquante
        self.result_df = self.result_df.dropna(subset=[self.config.datetime_column])

    # ---------- charge  ----------
    def load(self) -> pd.DataFrame:
        if self.result_df is None:
            raise ValueError("Pipeline not run yet.")
        return self.result_df

    # ---------- RUN EVERYTHING ----------
    def run(self) -> pd.DataFrame:
        self.extract()
        self.clean()
        self.transform()
        return self.load()


if __name__ == "__main__":
    config = ETLConfig(
        logs_path="logs_arche.csv",
        notes_path="notes.csv",
        course_column="contexte",      # à adapter à ton fichier
        course_value="NomDuCours",     # mettre le vrai nom du cours ou None
        datetime_column="heure",
        id_column="pseudo"
    )

    etl = ETLPipeline(config)
    df_clean = etl.run()
    print(df_clean.head())


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 13)